#  Speech Acoustic Emotion Recognition

## Feature Extraction

In [ ]:
# Basic imports
import os
import sys
import glob
import pickle
import warnings
import re
from dotenv import load_dotenv

# Data manipulation and numerical processing libraries
import numpy as np
import pandas as pd

# Audio processing libraries
import librosa
import librosa.display
import soundfile as sf
import opensmile

# Visualization library
import matplotlib.pyplot as plt

# Machine learning and evaluation tools
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Progress bar for loops
from tqdm.notebook import tqdm

# Ignore warnings for cleaner output
warnings.filterwarnings("ignore")

# Custom feature extraction modules from psychai (assuming psychai is installed in your environment)
import psychai.feature.feature_extraction.feature_retriever
import psychai.feature.feature_extraction.feature_processor
import psychai.speech_acoustics.feature_extraction.feature_extraction

# Initialize feature processor from psychai
feature_processor = psychai.feature.feature_extraction.feature_processor.FeatureProcessor()

# Set base directories and file paths

load_dotenv(override=True)
huggingface_cache_location = os.getenv("huggingface_cache_location")
datasets_cache_location = os.getenv("datasets_cache_location")
resutls_location = os.getenv("results_location")
resource_path= os.getenv("resources_location")

raw_data_path =  os.path.join(resource_path, "examples", "paper5_mmer_moral","resources","raw_data")
csv_cache_file_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "audio","audio_features_cache_20241216.csv")
csv_results_file_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "audio","audio_features_merged_20241216.csv")
pickle_file_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "audio","audio_features_cache_241204.pkl")
pickle_folder_path =  os.path.join(resutls_location, "examples", "paper5_mmer_moral", "audio")

# Extend system path to current directory (for importing modules if needed)
current_dir = os.path.abspath('../')
sys.path.append(current_dir)

# Import any custom utilities from the parent directory if needed
import local_utilities

# Specify modality (e.g., audio segment type) for processing
modality_value = "Segment_Audio"


#### Rename

If the raw data is not in the correct format, follow these steps to reformat it. If your data format is correct, you can skip this section.

In [ ]:
# for user_id in range(5,15):

#     # Format the user folder ID as a three-digit string (e.g., 001, 002, etc.)
#     user_folder = f"{user_id:03d}"

#     # Construct the full path to the modality folder inside the user's directory
#     modality_folder = os.path.join(base_directory, user_folder, modality_value)

#     if os.path.exists(modality_folder):
#         for file_path in os.listdir(modality_folder):
#             old_file_path = os.path.join(modality_folder, file_path)
#             # Regular expression to match the original file format
#             match = re.match(r'PS-9_(\d{3})_(\d{2})_(\d{4})-(\d{2})-(\d{2})_(\d{6})_(\d+)\.(.+)', file_path)
#             match_short = re.match(r'PS-9_(\d{3})_(\d{1})_(\d{4})-(\d{2})-(\d{2})_(\d{6})_(\d+)\.(.+)', file_path)
#             if match_short:
#                 match = match_short
#             if match:
#                 user_number = match.group(1)
#                 step = match.group(2)
#                 year = match.group(3)
#                 month = match.group(4)
#                 day = match.group(5)
#                 time = match.group(6)  # HHMMSS format
#                 sequence = match.group(7)
#                 extension = match.group(8)

#                 # Convert time from HHMMSS to individual components
#                 hour = time[:2]
#                 minute = time[2:4]
#                 second = time[4:]

#                 # Create new file name in the required format
#                 new_file_name = f"PS-9_{user_number}_{step}_{month}_{day}_{hour}_{minute}_{second}_{sequence}.{extension}"
#                 new_file_path = os.path.join(modality_folder, new_file_name)

#                 try:
#                     # Rename the file
#                     os.rename(old_file_path, new_file_path)
#                     print(f"Renamed: {old_file_path} -> {new_file_name}")
#                 except Exception as e:
#                     print(f"Error renaming file {file_path}: {e}")
#             else:
#                 print(f"Filename format does not match: {file_path}")
    

### Prepare data

In [ ]:
single_analysis = local_utilities.SingleModalityAnalysis()

filters = [(2, '25'), (2, '26'), (2, '27'), (2, '28')]

# consider user from 005~013
user_ids = range(11, 14)

# rct.csv contains user group information
#merging_pdf = pd.read_csv('./resources/data/rct/rct.csv')

group_info_path =  os.path.join(resource_path, "examples", "paper5_mmer_moral", "group_info","rct.csv")
merging_pdf = pd.read_csv(group_info_path)

df_largest = single_analysis.prosss_folder(raw_data_path , filters, user_ids, merging_pdf, modality_value)

### Single File Example

In [ ]:
# Load audio file
single_file_path = df_largest.iloc[0]["files"] 
y, sr = librosa.load(single_file_path, sr=None)

#### Waveform

In [ ]:
plt.figure(figsize=(14, 5))
librosa.display.waveshow(y, sr=sr)
plt.title("Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()

#### Spectrogram

In [ ]:
D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
plt.figure(figsize=(14, 5))
librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log', cmap='coolwarm')
plt.colorbar(format="%+2.0f dB")
plt.title("Log-frequency Power Spectrogram")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.show()

#### Mel-frequency cepstral coefficients (MFCCs)

In [ ]:
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
plt.figure(figsize=(14, 5))
librosa.display.specshow(mfccs, x_axis='time', sr=sr, cmap='viridis')
plt.colorbar()
plt.title("MFCC")
plt.xlabel("Time (s)")
plt.ylabel("MFCC Coefficients")
plt.show()

#### Chroma Features

In [ ]:
chroma = librosa.feature.chroma_stft(y=y, sr=sr)
plt.figure(figsize=(14, 5))
librosa.display.specshow(chroma, x_axis='time', y_axis='chroma', sr=sr, cmap='coolwarm')
plt.colorbar()
plt.title("Chroma Features")
plt.xlabel("Time (s)")
plt.ylabel("Chroma")
plt.show()

#### Zero-crossing rate

In [ ]:
zcr = librosa.feature.zero_crossing_rate(y)
plt.figure(figsize=(14, 5))
plt.plot(zcr[0])
plt.title("Zero Crossing Rate")
plt.xlabel("Frame")
plt.ylabel("Rate")
plt.show()

#### Spectral Centroid

In [ ]:
spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
frames = range(len(spectral_centroid))
times = librosa.frames_to_time(frames, sr=sr)
plt.figure(figsize=(14, 5))
plt.semilogy(times, spectral_centroid, label='Spectral Centroid')
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("Spectral Centroid")
plt.legend()
plt.show()

#### Spectral Bandwidth

In [ ]:
spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
plt.figure(figsize=(14, 5))
plt.semilogy(times, spectral_bandwidth, label='Spectral Bandwidth')
plt.xlabel("Time (s)")
plt.ylabel("Bandwidth (Hz)")
plt.title("Spectral Bandwidth")
plt.legend()
plt.show()

#### openSMILE GeMAPS

Eyben, F., Scherer, K. R., Schuller, B. W., Sundberg, J., André, E., Busso, C., ... & Truong, K. P. (2015). The Geneva minimalistic acoustic parameter set (GeMAPS) for voice research and affective computing. IEEE transactions on affective computing, 7(2), 190-202.

In [ ]:
import opensmile
import pandas as pd

# Initialize the openSMILE object with an emotion-relevant configuration
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.GeMAPS,  # Change to FeatureSet.IS09_emotion for IS09 features
    #feature_set=opensmile.FeatureSet.IS09_emotion,
    feature_level=opensmile.FeatureLevel.Functionals
)

# Extract features
features = smile.process_file(single_file_path)

# Display extracted features in a DataFrame
df_open_smile = features.reset_index()  # Convert index to columns for better readability

df_open_smile


#### Features

In [ ]:
# Extract features
features = {}

# 1. Zero-Crossing Rate
zcr = librosa.feature.zero_crossing_rate(y)[0]
features['zero_crossing_rate_mu'] = np.mean(zcr)
features['zero_crossing_rate_std'] = np.std(zcr)

# 2. Spectral Centroid
spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
features['spectral_centroid_mu'] = np.mean(spectral_centroid)
features['spectral_centroid_std'] = np.std(spectral_centroid)

# 3. Spectral Bandwidth
spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
features['spectral_bandwidth_mu'] = np.mean(spectral_bandwidth)
features['spectral_bandwidth_std'] = np.std(spectral_bandwidth)

# 4. MFCCs (Mel-frequency cepstral coefficients)
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
for i, mfcc in enumerate(mfccs):
    features[f'mfcc_{i+1}_mu'] = np.mean(mfcc)
    features[f'mfcc_{i+1}_std'] = np.std(mfcc)

# 5. Chroma Features
chroma = librosa.feature.chroma_stft(y=y, sr=sr)
for i, chroma_value in enumerate(chroma):
    features[f'chroma_{i+1}_mu'] = np.mean(chroma_value)
    features[f'chroma_{i+1}_std'] = np.std(chroma_value)

# 6. Spectral Contrast
spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
for i, contrast in enumerate(spectral_contrast):
    features[f'spectral_contrast_{i+1}_mu'] = np.mean(contrast)
    features[f'spectral_contrast_{i+1}_std'] = np.std(contrast)

# 7. Tonnetz (Tonal Centroids)
tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
for i, tonnetz_value in enumerate(tonnetz):
    features[f'tonnetz_{i+1}_mu'] = np.mean(tonnetz_value)
    features[f'tonnetz_{i+1}_std'] = np.std(tonnetz_value)

# Convert features to a DataFrame for better presentation
df_librosa = pd.DataFrame([features])

In [ ]:
combined_columns = pd.concat([df_librosa.reset_index(drop=True), df_open_smile.reset_index(drop=True)], axis=1)

In [ ]:
combined_columns

In [ ]:
speech_feature_extractor =  psychai.speech_acoustics.feature_extraction.feature_extraction.SpeechFeatureExtractor(segment_length=20, pick_chche_folder=pickle_folder_path)
speech_feature_extractor.segment_and_extract_features(single_file_path, extraction_methods=["librosa","opensmile"])

### Extract Features

In [ ]:
speech_feature_extractor =  psychai.speech_acoustics.feature_extraction.feature_extraction.SpeechFeatureExtractor(segment_length=20, pick_chche_folder=pickle_folder_path)

In [ ]:
df_largest

In [ ]:
# df_largest_trimmed = df_largest.copy()

# # Keep only the first row of the DataFrame
# df_largest_trimmed = df_largest.iloc[3:5]

# # Optionally reset the index if needed
# df_largest_trimmed.reset_index(drop=True, inplace=True)

In [ ]:
from functools import partial
processor = psychai.feature.feature_extraction.feature_retriever.FeatureRetriever(df_largest, feature_extractors=[partial(speech_feature_extractor.segment_and_extract_features, extraction_methods=["librosa"])]
                             ,cache_file_path=pickle_file_path)

In [ ]:
to_override = True 
if to_override == True:
    df_with_features = processor.extract_features(override= to_override)
    df_with_features.to_csv(csv_cache_file_path)
else:
    df_with_features = processor.extract_features(override= to_override)

In [ ]:
print(df_with_features)


In [ ]:
df_with_features.columns

In [ ]:
# Remove rows with at least one NaN
df_cleaned = df_with_features.dropna()
df_cleaned.reset_index(drop=True, inplace=True)

print("\nDataFrame after removing rows with at least one NaN:")
print(df_cleaned)

In [ ]:
df_cleaned["librosa_tonnetz_5"]

In [ ]:
df_summary = speech_feature_extractor.summarized_data(df_cleaned)
df_summary

In [ ]:
df_merged = pd.merge(df_cleaned, df_summary, on='files', how='left')
df_merged.to_csv(csv_results_file_path)

## Footnote
- Copyright (c) 2026 Ivan Liu. Licensed under CC BY 4.0.
- Last Update: 2024
- Env: env41124
- References: 
    - https://github.com/PrudhviGNV/Speech-Emotion-Recognization